# Extending Attributes

### Importing Functions

In [21]:
%cd "C:\Users\trish\Desktop\laser cut\Challange_1_LaserCut-GmbH\DSF4E_Challenge1-2"
import pandas as pd
import os

C:\Users\trish\Desktop\laser cut\Challange_1_LaserCut-GmbH\DSF4E_Challenge1-2


In [22]:
from helpers import (
    get_cutting_length,
    get_dxf_bounds,
    get_number_of_holes,
    load_dxf_geometry,
    plot_dxf
)

### Attribute Overview

Here, we read the cleaned customers_sales file, saved as cleaned_customer_sales.csv, as well as the folder containing all the DXF files

In [23]:
data_path = r"C:\Users\trish\Desktop\laser cut\Challange_1_LaserCut-GmbH\DSF4E_Challenge1-2\customers_sales_cleaned.csv"
dxf_folder = r"C:\Users\trish\Desktop\laser cut\Challange_1_LaserCut-GmbH\DSF4E_Challenge1-2\10991360"

sales = pd.read_csv(data_path)
print(sales.shape)
display(sales.head())
sales.tail()


(79, 5)


,customer_name,customer_branch,file_name,order_quantity,possible_duplicate
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,False
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,False
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,False
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,True
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,True


,customer_name,customer_branch,file_name,order_quantity,possible_duplicate
74,AutoParts AG,Munich Branch,Geo (78).DXF,8150,False
75,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,False
76,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,False
77,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,False
78,PrecisionCut GmbH,Stuttgart Branch,Geo (9).DXF,3750,False


### Extending Data Set

The functions in the helper document give us all the attributes that needed to be added. We put all the additional attributes in a set, and add it to the pre-existing cleaned data. 

In [24]:
def bounding_info(file_name):
    """Return bounding width, length, cutting length, number of holes, and bounding box area for one DXF file."""

    path = os.path.join(dxf_folder, file_name)

    width, length = get_dxf_bounds(path)

    if width is None or length is None:
        return pd.Series([None, None, None, None, None])

    number_of_holes = get_number_of_holes(path)
    cutting_length = get_cutting_length(path)

    if number_of_holes is None or cutting_length is None:
        return pd.Series([None, None, None, None, None])

    return pd.Series([
        width,
        length,
        cutting_length,
        number_of_holes,
        width * length
    ])


sales[[
    "bounding_width",
    "bounding_length",
    "cutting_length",
    "number_of_holes",
    "bounding_box_area"
]] = sales["file_name"].apply(bounding_info)

display(sales.head())


,customer_name,customer_branch,file_name,order_quantity,possible_duplicate,bounding_width,bounding_length,cutting_length,number_of_holes,bounding_box_area
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,False,2110.000062,2110.000062,3438.024977,0.0,4.452100e+06
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,False,853.825891,333.401400,3646.101350,0.0,2.846667e+05
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,False,164.020585,145.152365,1993.853699,0.0,2.380798e+04
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,True,193.800891,138.442589,1302.774283,0.0,2.683030e+04
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,True,937.150697,74.006665,2815.159795,9.0,6.935540e+04


Since everything seems to be working, we save the data in a new csv file.

In [25]:
# Save everything: the old columns plus the three new ones
sales.to_csv("customers_sales_with_bounds.csv", index=False)